*0.1 Python for GenAI*

# Break → Fix

**The situation.** A batch job grades 1,000 model answers overnight. It uses the async client, it has `await` everywhere, and it takes three hours. Nobody questions it — it works. Then the batch grows to 10,000 and the job no longer finishes before morning.

**The bug.** `await` inside a `for` loop. It *looks* concurrent. It runs one request at a time, because the loop waits for each answer before starting the next. Everything works, just ten times slower than it should, so it hides for months.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

import asyncio
from concurrent.futures import ThreadPoolExecutor


# A notebook already has an event loop running, so asyncio.run() is not allowed here.
# This helper runs the async code on a separate thread instead. In a normal script you
# would simply write asyncio.run(main()).
def run_async(coroutine):
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, coroutine).result()

**The broken version, timed.**

In [2]:
import time

from openai import AsyncOpenAI

answers_to_grade = []
for number in range(1, 13):
    answers_to_grade.append(
        f"Answer {number}: to reset your password, open Settings and choose Forgot password."
    )


async def grade(ai: AsyncOpenAI, answer: str) -> str:
    reply = await ai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "user", "content": f"Is this answer helpful? Reply yes or no.\n{answer}"}
        ],
        temperature=0,
        max_tokens=2,
    )
    return (
        reply.choices[0].message.content.strip().lower().strip(".")
    )  # "Yes." and "yes" are the same grade


async def broken() -> list[str]:
    async with AsyncOpenAI(timeout=30) as ai:
        grades = []
        for answer in answers_to_grade:
            grades.append(await grade(ai, answer))  # waits for each grade before starting the next
        return grades


started = time.perf_counter()
broken_grades = run_async(broken())
broken_seconds = time.perf_counter() - started
print("BREAK  await in a loop:", round(broken_seconds, 1), "s")

BREAK  await in a loop: 7.7 s


**The fix.** Prepare every request first, then `gather` them — with a semaphore so the fix does not turn a slow job into a rate-limited one.

In [3]:
async def fixed() -> list[str]:
    door = asyncio.Semaphore(6)
    async with AsyncOpenAI(timeout=30) as ai:

        async def guarded(answer: str) -> str:
            async with door:
                return await grade(ai, answer)

        tasks = []
        for answer in answers_to_grade:
            tasks.append(guarded(answer))
        return await asyncio.gather(*tasks)


started = time.perf_counter()
fixed_grades = run_async(fixed())
fixed_seconds = time.perf_counter() - started
print("FIX    gather + semaphore:", round(fixed_seconds, 1), "s")
print(
    "grades produced:",
    len(broken_grades),
    "and",
    len(fixed_grades),
    "— the model is not perfectly repeatable, so compare counts, not answers",
)
assert fixed_seconds < broken_seconds

FIX    gather + semaphore: 1.4 s
grades produced: 12 and 12 — the model is not perfectly repeatable, so compare counts, not answers


**Reading the output.** The same twelve grades, several times faster. The gap grows with the batch: at 10,000 answers it is the difference between hours and minutes.

**How you notice it.** Run time grows in a straight line with the number of items, while the CPU sits idle. When you see that, look for `await` inside a `for` loop over independent items.

**Watch out**
- Add the semaphore in the same change as `gather`, or the job fails with 429s instead of being slow.
- In code review, `await` directly inside a `for` over independent items is worth a comment every time — it is almost never what was meant.